In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into memory (keep full tables in variables)
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

def ts_min_max_info(df):
    """Return min/max timestamp (raw secs) and corresponding UTC+8 datetimes (compact)."""
    if 'timestamp' not in df.columns or df['timestamp'].dropna().empty:
        return {"min_ts": None, "min_dt_utc8": None, "max_ts": None, "max_dt_utc8": None}
    min_ts = int(df['timestamp'].min())
    max_ts = int(df['timestamp'].max())
    min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    return {"min_ts": min_ts, "min_dt_utc8": min_dt, "max_ts": max_ts, "max_dt_utc8": max_dt}

def top_unique_list(df, col, limit=20):
    """Return top unique values for a column (limited), or empty list if column missing."""
    if col not in df.columns:
        return []
    vals = pd.Series(df[col].dropna().unique())
    return vals.head(limit).tolist()

# Summaries per file
metric_summary = {
    "file": "metric.csv",
    "total_rows": int(metric.shape[0]),
    **ts_min_max_info(metric),
    "distinct_cmdb_id_top20": top_unique_list(metric, "cmdb_id", 20),
    "distinct_kpi_name_top20": top_unique_list(metric, "kpi_name", 20)
}

trace_summary = {
    "file": "trace.csv",
    "total_rows": int(trace.shape[0]),
    **ts_min_max_info(trace),
    "distinct_cmdb_id_top20": top_unique_list(trace, "cmdb_id", 20),
    "distinct_trace_name_top20": top_unique_list(trace, "trace_name", 20)
}

log_summary = {
    "file": "log.csv",
    "total_rows": int(log.shape[0]),
    **ts_min_max_info(log),
    "distinct_cmdb_id_top20": top_unique_list(log, "cmdb_id", 20),
    "distinct_log_name_top20": top_unique_list(log, "log_name", 20)
}

error_logs_summary = {
    "file": "error_logs.csv",
    "total_rows": int(error_logs.shape[0]),
    **ts_min_max_info(error_logs),
    "distinct_cmdb_id_top20": top_unique_list(error_logs, "cmdb_id", 20),
    "message_column_exists": "yes" if "message" in error_logs.columns else "no"
}

# Compact JSON-like dict per file as required (stored in variables and displayed)
metric_summary, trace_summary, log_summary, error_logs_summary

```
Out[1]:
```
```python
# Summary string (plain English) based on the previously computed summaries in kernel variables
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv: 144,804 rows; timestamps 1647772200 -> 1647773940 "
    "(UTC+8: 2022-03-20 18:30:00 -> 2022-03-20 18:59:00). "
    "Top cmdb_ids (up to 20) include: adservice, adservice-0, adservice-1, adservice-2, adservice2, cartservice, cartservice-0, cartservice-1, cartservice-2, ... . "
    "Top kpi_names (up to 20) include: app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, app.http.mrt, ...\n\n"
    "trace.csv: 36,928 rows; timestamps 1647772200 -> 1647773940 "
    "(UTC+8: 2022-03-20 18:30:00 -> 2022-03-20 18:59:00). "
    "Top cmdb_ids (up to 20) include: adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, ... . "
    "Top trace_names (up to 20) include: trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count, ...\n\n"
    "log.csv: 1,814 rows; timestamps 1647772200 -> 1647773940 "
    "(UTC+8: 2022-03-20 18:30:00 -> 2022-03-20 18:59:00). "
    "Top cmdb_ids (up to 20) include: adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, ... . "
    "Top log_names (up to 20) include: log.error_count, log.row_count.\n\n"
    "error_logs.csv: 2,351 rows; timestamps 1647772201 -> 1647773999 "
    "(UTC+8: 2022-03-20 18:30:01 -> 2022-03-20 18:59:59). "
    "Top cmdb_ids (up to 20) include: frontend-1, frontend-0, frontend-2. "
    "message column exists: yes.\n\n"
    "Notes: Lists of cmdb_ids and KPI/feature names above are truncated to top items (up to 20) for compactness."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'file': 'metric.csv', 'total_rows': 144804, 'min_ts': 1647772200, 'min_dt_utc8': '2022-03-20 18:30:00 CST', 'max_ts': 1647773940, 'max_dt_utc8': '2022-03-20 18:59:00 CST', 'distinct_cmdb_id_top20': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'distinct_kpi_name_top20': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact']}, {'file': 'trace.csv', 'total_rows': 36928, 'min_ts': 1647772200, 'min_dt_utc8': '2022-03-20 18:30:00 CST', 'max_ts': 1647773940, 'max_dt_utc8': '2022-03-20 18:59:00 CST', 'distinct_cmdb_id_top20': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'distinct_trace_name_top20': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count']}, {'file': 'log.csv', 'total_rows': 1814, 'min_ts': 1647772200, 'min_dt_utc8': '2022-03-20 18:30:00 CST', 'max_ts': 1647773940, 'max_dt_utc8': '2022-03-20 18:59:00 CST', 'distinct_cmdb_id_top20': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'distinct_log_name_top20': ['log.error_count', 'log.row_count']}, {'file': 'error_logs.csv', 'total_rows': 2351, 'min_ts': 1647772201, 'min_dt_utc8': '2022-03-20 18:30:01 CST', 'max_ts': 1647773999, 'max_dt_utc8': '2022-03-20 18:59:59 CST', 'distinct_cmdb_id_top20': ['frontend-1', 'frontend-0', 'frontend-2'], 'message_column_exists': 'yes'})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing metric DataFrame if present, otherwise load
if 'metric' not in globals():
    metric = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series
global_p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to the incident window [1647772200, 1647773940] inclusive
start_ts = 1647772200
end_ts = 1647773940
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) Merge window points with their global_p95 and mark anomalies (value > global_p95)
mw = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
mw['is_anom'] = mw['value'] > mw['global_p95']

# Aggregate counts per (cmdb_id, kpi_name)
agg_counts = (
    mw.groupby(['cmdb_id', 'kpi_name'], as_index=False)
      .agg(points_in_window=('value', 'count'),
           anomaly_count=('is_anom', 'sum'))
)

# Earliest anomaly timestamp per group (if any)
earliest_anom = (
    mw[mw['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anomaly_ts'})
)

# Combine results
result = (
    agg_counts
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
)

# Convert earliest_anomaly_ts to UTC+8 datetime strings
def ts_to_utc8_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_anomaly_dt_utc8'] = result['earliest_anomaly_ts'].apply(lambda x: ts_to_utc8_str(x) if pd.notna(x) else None)

# Ensure numeric types
result['anomaly_count'] = result['anomaly_count'].astype(int)
result['points_in_window'] = result['points_in_window'].astype(int)

# 4) Sorting and top-20 selection
if result['anomaly_count'].sum() == 0:
    # No anomalies: top 20 by points_in_window descending
    result_sorted = result.sort_values(by=['points_in_window', 'cmdb_id', 'kpi_name'], ascending=[False, True, True], na_position='last')
else:
    # Has anomalies: sort by anomaly_count desc, then earliest_anomaly_ts asc (NaNs last)
    result_sorted = result.sort_values(by=['anomaly_count', 'earliest_anomaly_ts', 'cmdb_id', 'kpi_name'],
                                       ascending=[False, True, True, True],
                                       na_position='last')

# Limit columns and rows for compact output
result_top20 = result_sorted[
    ['cmdb_id', 'kpi_name', 'global_p95', 'points_in_window', 'anomaly_count', 'earliest_anomaly_ts', 'earliest_anomaly_dt_utc8']
].head(20)

result_top20

```
Out[2]:
```
```python
# Plain-English summary of the computed anomaly aggregates (based on kernel-stored metric table)
summary = (
    "I computed the global P95 for every (cmdb_id, kpi_name) using the full metric series, "
    "filtered metric points to the incident window [1647772200, 1647773940] (UTC+8: 2022-03-20 18:30:00 -> 18:59:00), "
    "and counted window points exceeding the global P95 as anomalies.\n\n"
    "Top findings (top 20 rows returned):\n"
    "- All top entries show anomaly_count = 2 (i.e., two samples in the 30-minute window exceeded the global P95).\n"
    "- Most entries have points_in_window = 30 (one had 29).\n"
    "- The earliest anomaly timestamp for these top entries is 2022-03-20 18:30:00 (UTC+8).\n"
    "- Affected components/KPIs in the top results include adservice (CPU and JVM memory/pool KPIs), adservice2, cartservice-2, checkoutservice (app.grpc.mrt), currencyservice-2 (container memory KPIs), frontend-0/-1 (mesh request/response KPIs), and several istio/mesh-related KPIs.\n\n"
    "Interpretation: anomalies are present but sparse (only 2 out of ~30 points exceeded the global P95 for each highlighted KPI), and they occur at the start of the incident window. These candidates are good starting points for root-cause investigation, focusing on the listed components and the specific KPI types (CPU, JVM memory, container memory, and mesh request/response metrics)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  points_in_window  anomaly_count  earliest_anomaly_ts earliest_anomaly_dt_utc8
245            adservice    runtime.java_lang_OperatingSystem_SystemCpuLoad  1.724596e-02                30              2         1.647772e+09  2022-03-20 18:30:00 CST
291            adservice                    runtime.jvm_memory_MB_used.heap  2.059627e+01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
313            adservice         runtime.jvm_memory_pool_MB_used.Eden_Space  4.747099e+00                30              2         1.647772e+09  2022-03-20 18:30:00 CST
424          adservice-0  mesh.source.adservice.jaeger-collector.istio_r...  5.718750e-03                30              2         1.647772e+09  2022-03-20 18:30:00 CST
822           adservice2  runtime.java_lang_OperatingSystem_FreePhysical...  1.039938e+08                30              2         1.647772e+09  2022-03-20 18:30:00 CST
828           adservice2    runtime.java_lang_OperatingSystem_SystemCpuLoad  3.004202e-02                30              2         1.647772e+09  2022-03-20 18:30:00 CST
863           adservice2                         runtime.jvm_classes_loaded  9.166667e-02                30              2         1.647772e+09  2022-03-20 18:30:00 CST
994         adservice2-0  mesh.destination.frontend2.adservice2.istio_re...  2.126313e-01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
1279       cartservice-2  mesh.destination.frontend.cartservice.istio_re...  1.566325e-01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
1383     checkoutservice                                       app.grpc.mrt  5.743793e-02                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2050   currencyservice-2              container.node-5.container_memory_rss  3.292078e+07                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2052   currencyservice-2         container.node-5.container_memory_usage_MB  6.533398e+01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2053   currencyservice-2   container.node-5.container_memory_working_set_MB  6.001758e+01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2118  currencyservice2-0            container.node-6.container_memory_cache  6.496584e+07                29              2         1.647772e+09  2022-03-20 18:30:00 CST
2570          frontend-0  mesh.source.frontend.cartservice.istio_request...  1.750788e-01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2695          frontend-1  mesh.destination.unknown.frontend.istio_reques...  3.705988e+04                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2696          frontend-1  mesh.destination.unknown.frontend.istio_reques...  8.522500e+03                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2699          frontend-1  mesh.destination.unknown.frontend.istio_reques...  4.910988e+00                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2700          frontend-1  mesh.destination.unknown.frontend.istio_reques...  1.009713e-01                30              2         1.647772e+09  2022-03-20 18:30:00 CST
2707          frontend-1  mesh.destination.unknown.frontend.istio_respon...  1.084998e+06                30              2         1.647772e+09  2022-03-20 18:30:00 CST```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables if present
if 'metric' not in globals():
    metric = pd.read_csv("metric.csv")

# Ensure global_p95 exists (computed across full series)
if 'global_p95' not in globals():
    global_p95 = (
        metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

tz = pytz.timezone('Asia/Shanghai')
start_ts = 1647772200
end_ts = 1647773940

# Candidate list as requested
candidates = [
    ("adservice", "runtime.java_lang_OperatingSystem_SystemCpuLoad"),
    ("adservice", "runtime.jvm_memory_MB_used.heap"),
    ("adservice", "runtime.jvm_memory_pool_MB_used.Eden_Space"),
    ("adservice2", "runtime.java_lang_OperatingSystem_SystemCpuLoad"),
    ("adservice2", "runtime.jvm_classes_loaded"),
    ("checkoutservice", "app.grpc.mrt"),
    ("currencyservice-2", "container.node-5.container_memory_rss"),
    ("currencyservice-2", "container.node-5.container_memory_usage_MB"),
    ("currencyservice-2", "container.node-5.container_memory_working_set_MB"),
    ("currencyservice2-0", "container.node-6.container_memory_cache"),
    # frontend-0 and frontend-1: include their non-mesh KPIs dynamically below
]

# Discover non-mesh KPIs for frontend-0 and frontend-1 and append
for fe in ["frontend-0", "frontend-1"]:
    kpis = metric[(metric['cmdb_id'] == fe) & (~metric['kpi_name'].str.startswith('mesh.', na=False))]['kpi_name'].unique()
    # if any, add each as candidate
    for k in kpis:
        candidates.append((fe, k))

# Deduplicate candidates preserving order
seen = set()
candidates_unique = []
for c in candidates:
    if c not in seen:
        seen.add(c)
        candidates_unique.append(c)

# Prepare an empty list to collect fault runs
runs = []

# Helper to convert ts to UTC+8 string
def ts_to_utc8_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Process each candidate
for cmdb_id, kpi_name in candidates_unique:
    # Get global_p95 for this pair
    row = global_p95[(global_p95['cmdb_id'] == cmdb_id) & (global_p95['kpi_name'] == kpi_name)]
    if row.empty:
        # skip if no global threshold available
        continue
    gp95 = float(row['global_p95'].iloc[0])
    # Extract window points for this (cmdb_id,kpi_name)
    dfw = metric[
        (metric['cmdb_id'] == cmdb_id) &
        (metric['kpi_name'] == kpi_name) &
        (metric['timestamp'] >= start_ts) &
        (metric['timestamp'] <= end_ts)
    ].copy()
    if dfw.empty:
        continue
    dfw = dfw.sort_values('timestamp').reset_index(drop=True)
    # Mark anomaly points
    dfw['is_anom'] = dfw['value'] > gp95
    if not dfw['is_anom'].any():
        continue
    # Identify consecutive runs where is_anom==True and timestamps are consecutive by 60s
    # We create a group id that increments when the consecutive condition breaks
    dfw['ts_diff'] = dfw['timestamp'].diff().fillna(0).astype(int)
    # For building run groups we consider rows that are anomalies. We'll iterate to detect runs.
    run_id = 0
    current_run = None
    for idx, r in dfw.iterrows():
        if not r['is_anom']:
            current_run = None
            continue
        if current_run is None:
            # start new run
            run_id += 1
            current_run = {'run_id': run_id, 'start_idx': idx, 'end_idx': idx}
        else:
            # check timestamp continuity: current row ts should be previous ts + 60
            prev_ts = dfw.at[idx-1, 'timestamp']
            if r['timestamp'] - prev_ts == 60:
                current_run['end_idx'] = idx
            else:
                # finalize previous run and start new one
                s = current_run['start_idx']; e = current_run['end_idx']
                run_rows = dfw.loc[s:e]
                max_val = float(run_rows['value'].max())
                duration = int(len(run_rows))
                start_ts_run = int(run_rows['timestamp'].iloc[0])
                end_ts_run = int(run_rows['timestamp'].iloc[-1])
                severity_pct = None
                if gp95 != 0 and not np.isnan(gp95):
                    severity_pct = (max_val - gp95) / gp95 * 100.0
                runs.append({
                    'cmdb_id': cmdb_id,
                    'kpi_name': kpi_name,
                    'global_p95': gp95,
                    'fault_start_ts': start_ts_run,
                    'fault_end_ts': end_ts_run,
                    'duration_points': duration,
                    'max_value': max_val,
                    'severity_pct': None if severity_pct is None else float(severity_pct),
                    'earliest_fault_dt_utc8': ts_to_utc8_str(start_ts_run)
                })
                # start new run
                run_id += 1
                current_run = {'run_id': run_id, 'start_idx': idx, 'end_idx': idx}
    # finalize any ongoing run
    if current_run is not None:
        s = current_run['start_idx']; e = current_run['end_idx']
        run_rows = dfw.loc[s:e]
        max_val = float(run_rows['value'].max())
        duration = int(len(run_rows))
        start_ts_run = int(run_rows['timestamp'].iloc[0])
        end_ts_run = int(run_rows['timestamp'].iloc[-1])
        severity_pct = None
        if gp95 != 0 and not np.isnan(gp95):
            severity_pct = (max_val - gp95) / gp95 * 100.0
        runs.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'global_p95': gp95,
            'fault_start_ts': start_ts_run,
            'fault_end_ts': end_ts_run,
            'duration_points': duration,
            'max_value': max_val,
            'severity_pct': None if severity_pct is None else float(severity_pct),
            'earliest_fault_dt_utc8': ts_to_utc8_str(start_ts_run)
        })

# Create DataFrame of runs
if runs:
    runs_df = pd.DataFrame(runs)
else:
    runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','fault_start_ts','fault_end_ts',
        'duration_points','max_value','severity_pct','earliest_fault_dt_utc8'
    ])

# Filter to duration_points >=2, if none meet, relax to >=1
if not runs_df.empty and (runs_df['duration_points'] >= 2).any():
    filtered = runs_df[runs_df['duration_points'] >= 2].copy()
else:
    filtered = runs_df.copy()  # possibly empty or include single-point runs

# Sort by severity_pct desc (NaNs last), then earliest start ts asc
# For sorting, replace None with -inf so they go to bottom when descending
if not filtered.empty:
    filtered['severity_sort'] = filtered['severity_pct'].apply(lambda x: -1e18 if x is None else x)
    # We want descending severity_pct -> sort by severity_sort descending -> equivalently sort by severity_pct desc, but None last
    filtered = filtered.sort_values(
        by=['severity_pct', 'fault_start_ts'],
        ascending=[False, True],
        na_position='last'
    )
    filtered = filtered.drop(columns=['severity_sort'], errors='ignore')

# Limit to top 20 and select columns
fault_runs_top20 = filtered[
    ['cmdb_id','kpi_name','global_p95','fault_start_ts','fault_end_ts','duration_points','max_value','severity_pct','earliest_fault_dt_utc8']
].head(20)

fault_runs_top20

```
Out[3]:
```
Summary of detected fault runs for the requested non-mesh candidates (incident window 2022-03-20 18:30:00 → 18:59:00 UTC+8):

- Only frontend-0 and frontend-1 produced consecutive anomaly runs of duration ≥2 points:
  - frontend-0, container.node-5.container_fs_usage_MB./dev/vda1
    - global_p95 = 109.608 MB
    - fault window: 2022-03-20 18:58:00 → 18:59:00 (2 consecutive points)
    - max_value = 109.8555 MB, severity ≈ 0.23%
  - frontend-1, container.node-5.container_fs_usage_MB./dev/vda1
    - global_p95 = 115.4934 MB
    - fault window: 2022-03-20 18:58:00 → 18:59:00 (2 consecutive points)
    - max_value = 115.6953 MB, severity ≈ 0.17%
  - frontend-0, container.node-5.container_last_seen
    - global_p95 = 1647774000 (timestamp-like), fault 18:58 → 18:59, duration 2, negligible severity (~0.000005%)
  - frontend-1, container.node-5.container_last_seen
    - same pattern as frontend-0, negligible severity

- None of the explicitly listed non-mesh components (adservice, adservice2, checkoutservice, currencyservice-2, currencyservice2-0) produced multi-point fault runs (duration ≥2) in the incident window.

Interpretation / next steps:
- The only multi-point anomalies are brief (2 minutes) and low severity (fractional percent increases) and occur at the end of the window (18:58–18:59 UTC+8), primarily on frontend filesystem usage metrics and last-seen timestamps.
- These are low-impact candidates; if you need broader coverage, relax the duration threshold to include single-point spikes or re-examine other KPIs/time ranges.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name    global_p95  fault_start_ts  fault_end_ts  duration_points     max_value  severity_pct   earliest_fault_dt_utc8
0  frontend-0  container.node-5.container_fs_usage_MB./dev/vda1  1.096080e+02      1647773880    1647773940                2  1.098555e+02      0.225769  2022-03-20 18:58:00 CST
2  frontend-1  container.node-5.container_fs_usage_MB./dev/vda1  1.154934e+02      1647773880    1647773940                2  1.156953e+02      0.174861  2022-03-20 18:58:00 CST
1  frontend-0              container.node-5.container_last_seen  1.647774e+09      1647773880    1647773940                2  1.647774e+09      0.000005  2022-03-20 18:58:00 CST
3  frontend-1              container.node-5.container_last_seen  1.647774e+09      1647773880    1647773940                2  1.647774e+09      0.000005  2022-03-20 18:58:00 CST```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing trace DataFrame if present
if 'trace' not in globals():
    trace = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')
start_ts = 1647772200
end_ts = 1647773940

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full trace series
trace_quantiles = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(global_P95=('value', lambda x: float(x.quantile(0.95))),
         global_P05=('value', lambda x: float(x.quantile(0.05))))
)

# 2) Filter trace to incident window
trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# 3) Merge windowed data with global quantiles
tw = trace_window.merge(trace_quantiles, on=['cmdb_id', 'trace_name'], how='left')

# Identify rule types based on trace_name
name_series = tw['trace_name'].astype(str)
mask_high = name_series.str.contains(r'duration|p95|error', case=False, regex=True)
mask_low = name_series.str.contains(r'row_count|\.count', case=False, regex=True)

# Determine anomalies per rules
tw['is_anom'] = False
# high anomalies: value > global_P95
tw.loc[mask_high, 'is_anom'] = tw.loc[mask_high, 'value'] > tw.loc[mask_high, 'global_P95']
# low anomalies: value < global_P05
tw.loc[mask_low, 'is_anom'] = tw.loc[mask_low, 'value'] < tw.loc[mask_low, 'global_P05']
# If a name matches both masks (rare), consider either condition true -> already handled by above assignments

# 4) Aggregate per (cmdb_id, trace_name)
agg = (
    tw.groupby(['cmdb_id', 'trace_name'], as_index=False)
      .agg(global_P95=('global_P95', 'first'),
           global_P05=('global_P05', 'first'),
           points_in_window=('value', 'count'),
           anomaly_count=('is_anom', 'sum'))
)

# earliest anomaly timestamp
earliest = (
    tw[tw['is_anom']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anomaly_ts'})
)

result = agg.merge(earliest, on=['cmdb_id', 'trace_name'], how='left')

# Convert earliest anomaly ts to UTC+8 datetime string
def ts_to_utc8_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_anomaly_dt_utc8'] = result['earliest_anomaly_ts'].apply(lambda x: ts_to_utc8_str(x) if pd.notna(x) else None)

# Ensure numeric types
result['points_in_window'] = result['points_in_window'].astype(int)
result['anomaly_count'] = result['anomaly_count'].astype(int)

# 5) Sorting and top-20 selection
if result['anomaly_count'].sum() == 0:
    # No anomalies: top 20 by points_in_window descending
    result_sorted = result.sort_values(by=['points_in_window', 'cmdb_id', 'trace_name'], ascending=[False, True, True], na_position='last')
else:
    # Has anomalies: sort by anomaly_count desc, then earliest_anomaly_ts asc (NaNs last)
    result_sorted = result.sort_values(by=['anomaly_count', 'earliest_anomaly_ts', 'cmdb_id', 'trace_name'],
                                      ascending=[False, True, True, True],
                                      na_position='last')

trace_anomaly_summary_top20 = result_sorted[
    ['cmdb_id', 'trace_name', 'global_P95', 'global_P05', 'points_in_window', 'anomaly_count', 'earliest_anomaly_ts', 'earliest_anomaly_dt_utc8']
].head(20)

trace_anomaly_summary_top20

```
Out[4]:
```
Summary of trace anomaly scan (computed global P95/P05 across the full series; window = 2022-03-20 18:30:00 → 18:59:00 UTC+8):

- Method: for each (cmdb_id, trace_name) I compared window points to the global thresholds:
  - duration / p95 / error → anomaly if value > global_P95
  - row_count or .count → anomaly if value < global_P05

- Top findings (returned top 20): many trace series show anomaly_count = 2 (i.e., two samples in the 30-minute window triggered the rule), points_in_window = 30, and the earliest anomaly for those series is 2022-03-20 18:30:00 (UTC+8).

- Notable affected trace series (examples from the top rows):
  - adservice-0, adservice-2: trace.from_frontend-1.row_count (row_count below global P05)
  - cartservice-0 / cartservice-2: various trace.from_*.duration_mean and row_count series (duration spikes and row_count drops)
  - checkoutservice (0/1/2): several trace.to_shippingservice / trace.from_frontend / trace.to_cartservice row_count and duration_p95 series (duration/p95 increases and row_count drops)
  - checkoutservice2-0: trace.from_root.duration_mean and duration_p95 (duration increases)
  - checkoutservice2-0: trace.to_currencyservice2-0.duration_mean/p95 (duration increases)
  - currencyservice-0 and currencyservice-1: trace.from_frontend / trace.from_checkoutservice durations and row_count issues

- Pattern and interpretation:
  - The anomalies are sparse but consistent across multiple services: each flagged series has 2 anomalous samples at the start of the window (18:30:00).
  - Two main types appear: (a) decreased row_count (traffic/throughput drops) and (b) increased durations/p95 (latency spikes).
  - This suggests a short, system-wide transient at the beginning of the incident window affecting request rates and latencies across services.

- Suggested next steps:
  - Investigate logs/traces around 18:30:00 UTC+8 for the listed services (adservice*, cartservice*, checkoutservice*, checkoutservice2-0, currencyservice*).
  - Correlate with infrastructure metrics (network/node/pod metrics) and error logs to find a common upstream cause (traffic cut, gateway issue, or brief spike in latency).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  global_P95  global_P05  points_in_window  anomaly_count  earliest_anomaly_ts earliest_anomaly_dt_utc8
7           adservice-0             trace.from_frontend-1.row_count   26.550000   19.450000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
31          adservice-2             trace.from_frontend-1.row_count   26.550000   18.900000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
51        cartservice-0      trace.from_checkoutservice-2.row_count    4.000000    2.450000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
60        cartservice-0         trace.from_frontend-2.duration_mean    0.000049    0.000000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
108       cartservice-2         trace.from_frontend-0.duration_mean    0.000073    0.000000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
116       cartservice-2         trace.from_frontend-2.duration_mean    0.000063    0.000000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
120       cartservice-2                    trace.self.duration_mean    0.000012    0.000000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
224   checkoutservice-0    trace.to_shippingservice-2.duration_mean    0.000065    0.000054                30              2         1.647772e+09  2022-03-20 18:30:00 CST
225   checkoutservice-0     trace.to_shippingservice-2.duration_p95    0.000072    0.000058                30              2         1.647772e+09  2022-03-20 18:30:00 CST
313   checkoutservice-1     trace.to_shippingservice-1.duration_p95    0.000077    0.000056                30              2         1.647772e+09  2022-03-20 18:30:00 CST
325   checkoutservice-2          trace.from_frontend-1.duration_p95    0.090988    0.028195                30              2         1.647772e+09  2022-03-20 18:30:00 CST
343   checkoutservice-2            trace.to_cartservice-0.row_count    4.000000    2.450000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
356   checkoutservice-2    trace.to_currencyservice-1.duration_mean    0.000109    0.000091                30              2         1.647772e+09  2022-03-20 18:30:00 CST
357   checkoutservice-2     trace.to_currencyservice-1.duration_p95    0.000131    0.000100                30              2         1.647772e+09  2022-03-20 18:30:00 CST
416  checkoutservice2-0               trace.from_root.duration_mean    0.020111    0.005506                30              2         1.647772e+09  2022-03-20 18:30:00 CST
417  checkoutservice2-0                trace.from_root.duration_p95    0.056093    0.014168                30              2         1.647772e+09  2022-03-20 18:30:00 CST
428  checkoutservice2-0   trace.to_currencyservice2-0.duration_mean    0.003016    0.000377                30              2         1.647772e+09  2022-03-20 18:30:00 CST
429  checkoutservice2-0    trace.to_currencyservice2-0.duration_p95    0.009375    0.001060                30              2         1.647772e+09  2022-03-20 18:30:00 CST
463   currencyservice-0             trace.from_frontend-0.row_count  170.200000  113.600000                30              2         1.647772e+09  2022-03-20 18:30:00 CST
480   currencyservice-1  trace.from_checkoutservice-2.duration_mean    0.000109    0.000091                30              2         1.647772e+09  2022-03-20 18:30:00 CST```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing error_logs if present; otherwise load
if 'error_logs' not in globals():
    error_logs = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

# Incident sub-interval: 2022-03-20 18:30:00 -> 18:35:00 UTC+8
start_ts = 1647772200
end_ts = 1647772300

# 1) Filter error logs to the given interval and store full filtered table in kernel variable
error_logs_window = error_logs[(error_logs['timestamp'] >= start_ts) & (error_logs['timestamp'] <= end_ts)].copy()

# Helper to convert timestamp to UTC+8 datetime string
def ts_to_utc8_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 2) Aggregate per cmdb_id: count, earliest ts, earliest dt, and up to 3 distinct sample messages
if 'message' in error_logs_window.columns:
    samples = (
        error_logs_window
        .groupby('cmdb_id')['message']
        .apply(lambda s: '; '.join(pd.Series(s.dropna().unique()[:3])))
        .reset_index(name='sample_messages')
    )
else:
    # If no message column, produce empty samples column
    samples = pd.DataFrame({'cmdb_id': error_logs_window['cmdb_id'].unique(), 'sample_messages': None})

agg = (
    error_logs_window
    .groupby('cmdb_id', as_index=False)
    .agg(error_count=('timestamp', 'count'),
         earliest_error_ts=('timestamp', 'min'))
)

agg = agg.merge(samples, on='cmdb_id', how='left')

# Add UTC+8 converted earliest datetime
agg['earliest_error_dt_utc8'] = agg['earliest_error_ts'].apply(lambda x: ts_to_utc8_str(x) if pd.notna(x) else None)

# Sort by error_count desc, then earliest_error_ts asc; limit to top 20
error_agg_top20 = agg.sort_values(by=['error_count', 'earliest_error_ts'], ascending=[False, True]).head(20)

# Keep only the requested columns in compact form
error_agg_top20 = error_agg_top20[['cmdb_id', 'error_count', 'earliest_error_ts', 'earliest_error_dt_utc8', 'sample_messages']]

# Display the compact aggregate (full filtered table is stored in error_logs_window)
error_agg_top20

```
Out[5]:
```
```python
# Plain-English summary of error_logs scan (interval 2022-03-20 18:30:00 → 18:35:00 UTC+8)
summary = (
    "Error log summary for 2022-03-20 18:30:00 → 18:35:00 (UTC+8):\n"
    "- frontend-0: 45 errors, earliest 2022-03-20 18:30:01. Sample messages: 'severity: warning, message: failed to retrieve...'\n"
    "- frontend-2: 41 errors, earliest 2022-03-20 18:30:02. Sample messages: 'severity: warning, message: failed to retrieve...'\n"
    "- frontend-1: 37 errors, earliest 2022-03-20 18:30:01. Sample messages: 'severity: warning, message: failed to retrieve...'\n\n"
    "These three frontends are the top sources of error logs in the interval and share similar sample messages."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  error_count  earliest_error_ts   earliest_error_dt_utc8                                    sample_messages
0  frontend-0           45         1647772201  2022-03-20 18:30:01 CST  severity: warning, message: failed to retrieve...
2  frontend-2           41         1647772202  2022-03-20 18:30:02 CST  severity: warning, message: failed to retrieve...
1  frontend-1           37         1647772201  2022-03-20 18:30:01 CST  severity: warning, message: failed to retrieve...```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse kernel-stored metric DataFrame
# (assumes 'metric' exists in globals from prior steps)
tz = pytz.timezone('Asia/Shanghai')
start_ts = 1647772200
end_ts = 1647773940

# 1) Identify node cmdb_id values (start with 'node', case-insensitive)
all_cmdb = metric['cmdb_id'].dropna().astype(str).unique()
node_ids = [c for c in all_cmdb if str(c).lower().startswith('node')]

# Prepare output variable in case no nodes found
node_summary_top20 = None

if len(node_ids) == 0:
    # No node cmdb_id found -> empty list indicator
    node_summary_top20 = []
else:
    # 2) Select KPIs whose kpi_name contains any of the keywords (case-insensitive)
    keywords = ['cpu','systemcpu','memory','mem','disk','io','fs','load','swap']
    # build regex pattern
    pattern = r'(' + r'|'.join(re.escape(k) for k in keywords) + r')'
    # Filter metric for node cmdb_ids and kpi_name matching the pattern
    node_metric = metric[metric['cmdb_id'].isin(node_ids)].copy()
    node_metric = node_metric[node_metric['kpi_name'].astype(str).str.contains(pattern, case=False, na=False)].copy()
    
    if node_metric.empty:
        node_summary_top20 = []
    else:
        # 3) Compute global P95 per (node_cmdb_id, kpi_name) using full series (must be global)
        # Reuse existing global_p95 if available, else compute
        if 'global_p95' in globals():
            gp_all = global_p95.copy()
            # ensure columns are named consistently
            if 'global_p95' not in gp_all.columns:
                gp_all = gp_all.rename(columns={gp_all.columns[-1]: 'global_p95'})
        else:
            gp_all = (
                metric
                .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
                .quantile(0.95)
                .rename(columns={'value':'global_p95'})
            )
        # Filter gp_all for only node KPIs of interest
        gp_node = gp_all[gp_all['cmdb_id'].isin(node_ids) & gp_all['kpi_name'].astype(str).str.contains(pattern, case=False, na=False)].copy()
        # If gp_node is empty, nothing to check
        if gp_node.empty:
            node_summary_top20 = []
        else:
            # 3b) Filter to incident window and join with global P95
            node_window = node_metric[(node_metric['timestamp'] >= start_ts) & (node_metric['timestamp'] <= end_ts)].copy()
            # Merge to get global_p95 per series
            node_window = node_window.merge(gp_node, on=['cmdb_id','kpi_name'], how='left')
            # Drop rows without a global_p95 (safety)
            node_window = node_window[~node_window['global_p95'].isna()].copy()
            if node_window.empty:
                node_summary_top20 = []
            else:
                # Mark anomaly points: value > global_p95
                node_window['is_anom'] = node_window['value'] > node_window['global_p95']
                
                # Per (node,kpi) aggregate: anomaly_points count and whether >=1
                per_kpi = (
                    node_window.groupby(['cmdb_id','kpi_name'], as_index=False)
                    .agg(global_p95=('global_p95','first'),
                         points_in_window=('value','count'),
                         anomaly_points=('is_anom','sum'),
                         earliest_anom_ts=('timestamp', lambda x: int(x[node_window.loc[x.index,'is_anom']].min()) if (node_window.loc[x.index,'is_anom'].any()) else np.nan)
                        )
                )
                # Note: the lambda above is a bit complex due to groupby; safer to compute earliest separately
                # Recompute per_kpi more robustly:
                per_kpi = (
                    node_window.groupby(['cmdb_id','kpi_name'], as_index=False)
                    .agg(global_p95=('global_p95','first'),
                         points_in_window=('value','count'),
                         anomaly_points=('is_anom','sum'))
                )
                earliest_kpi = (
                    node_window[node_window['is_anom']]
                    .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
                    .min()
                    .rename(columns={'timestamp':'earliest_anom_ts'})
                )
                per_kpi = per_kpi.merge(earliest_kpi, on=['cmdb_id','kpi_name'], how='left')
                
                # 4) Per-node summary aggregation
                # total_kpis_checked: number of distinct kpi_names checked for that node (from gp_node)
                kpis_checked = gp_node.groupby('cmdb_id', as_index=False)['kpi_name'].nunique().rename(columns={'kpi_name':'total_kpis_checked'})
                
                # kpi_with_anomaly_count and total_anomaly_points and earliest_anomaly_ts across node
                per_node_anoms = (
                    per_kpi.groupby('cmdb_id', as_index=False)
                    .agg(kpi_with_anomaly_count=('anomaly_points', lambda s: int((s>0).sum())),
                         total_anomaly_points=('anomaly_points', 'sum'),
                         earliest_anomaly_ts=('earliest_anom_ts', 'min'))
                )
                # Up to 3 example kpi_names that had anomalies
                example_kpis = (
                    per_kpi[per_kpi['anomaly_points']>0]
                    .groupby('cmdb_id')['kpi_name']
                    .apply(lambda s: '; '.join(pd.Series(s.unique()[:3])))
                    .reset_index(name='example_kpis')
                )
                
                # Combine summaries
                node_summary = kpis_checked.merge(per_node_anoms, on='cmdb_id', how='left').merge(example_kpis, on='cmdb_id', how='left')
                # Fill NaNs with zeros / None where appropriate
                node_summary['kpi_with_anomaly_count'] = node_summary['kpi_with_anomaly_count'].fillna(0).astype(int)
                node_summary['total_anomaly_points'] = node_summary['total_anomaly_points'].fillna(0).astype(int)
                # earliest_anomaly_ts may be NaN -> keep as NaN
                node_summary['earliest_anomaly_dt_utc8'] = node_summary['earliest_anomaly_ts'].apply(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z') if not pd.isna(x) else None)
                
                # Select and rename columns as requested
                node_summary = node_summary.rename(columns={'cmdb_id':'node_cmdb_id'})
                node_summary = node_summary[['node_cmdb_id','total_kpis_checked','kpi_with_anomaly_count','total_anomaly_points','earliest_anomaly_ts','earliest_anomaly_dt_utc8','example_kpis']]
                
                # 5) Sort by total_anomaly_points desc and return top 20
                node_summary_top20 = node_summary.sort_values(by=['total_anomaly_points','node_cmdb_id'], ascending=[False, True]).head(20).reset_index(drop=True)

# Final compact result stored in node_summary_top20 (and intermediate tables: node_metric, gp_node, node_window, per_kpi exist)
node_summary_top20

```
Out[6]:
```
```python
summary = (
    "Node-level KPI anomaly summary (incident window 2022-03-20 18:30:00 → 18:59:00 UTC+8):\n\n"
    "Top nodes by total anomaly points:\n"
    "- node-2: checked 38 KPIs, 28 KPIs had ≥1 anomaly, total anomaly points = 53, earliest anomaly 2022-03-20 18:30:00.\n"
    "- node-1: checked 38 KPIs, 27 KPIs had ≥1 anomaly, total anomaly points = 49, earliest anomaly 2022-03-20 18:30:00.\n"
    "- node-6: checked 38 KPIs, 26 KPIs had ≥1 anomaly, total anomaly points = 49, earliest anomaly 2022-03-20 18:30:00.\n"
    "- node-3: checked 38 KPIs, 24 KPIs had ≥1 anomaly, total anomaly points = 44, earliest anomaly 2022-03-20 18:30:00.\n"
    "- node-4: checked 38 KPIs, 24 KPIs had ≥1 anomaly, total anomaly points = 40, earliest anomaly 2022-03-20 18:30:00.\n"
    "- node-5: checked 38 KPIs, 22 KPIs had ≥1 anomaly, total anomaly points = 37, earliest anomaly 2022-03-20 18:30:00.\n\n"
    "Common anomalous KPI examples include CPU-related metrics (system.cpu.iowait, system.cpu.pct_usage, system.cpu.*), indicating node-level CPU/iowait activity at the start of the window.\n\n"
    "Interpretation: all nodes show anomalies beginning at 18:30:00 UTC+8, with node-2 having the most anomaly points. Focus initial investigation on node CPU/iowait and I/O subsystem around 18:30 to find a common cause."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

node_cmdb_id  total_kpis_checked  kpi_with_anomaly_count  total_anomaly_points  earliest_anomaly_ts earliest_anomaly_dt_utc8                                       example_kpis
0       node-2                  38                      28                    53         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...
1       node-1                  38                      27                    49         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...
2       node-6                  38                      26                    49         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...
3       node-3                  38                      24                    44         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...
4       node-4                  38                      24                    40         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...
5       node-5                  38                      22                    37         1.647772e+09  2022-03-20 18:30:00 CST  system.cpu.iowait; system.cpu.pct_usage; syste...```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse kernel-stored variables
# metric (full series) and global_p95 (per-series global 95th) are expected to exist
# If not present, load/compute accordingly (but per instructions they exist)
tz = pytz.timezone('Asia/Shanghai')
start_ts = 1647772200
end_ts = 1647773940

nodes = ["node-2", "node-1", "node-6"]
keywords = ['cpu','iowait','load','mem','memory','disk','fs','io','swap']
pattern = r'(' + r'|'.join(re.escape(k) for k in keywords) + r')'

# 1) Select relevant KPIs for the specified nodes using full metric table
node_kpis_df = (
    metric[
        metric['cmdb_id'].isin(nodes) &
        metric['kpi_name'].astype(str).str.contains(pattern, case=False, na=False)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 2) Attach global_P95 from precomputed global_p95 (computed on full series earlier)
# Ensure global_p95 variable exists
if 'global_p95' not in globals():
    gp_all = (
        metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .quantile(0.95).rename(columns={'value':'global_p95'})
    )
else:
    gp_all = global_p95.copy()

# Merge to get gp95 for node KPIs
node_kpis_with_gp = node_kpis_df.merge(gp_all, on=['cmdb_id','kpi_name'], how='left')

# Filter out series without a global_p95
node_kpis_with_gp = node_kpis_with_gp[~node_kpis_with_gp['global_p95'].isna()].reset_index(drop=True)

# 2b) Filter full metric to the incident window for these nodes and KPIs and store full filtered table
node_window = metric[
    (metric['cmdb_id'].isin(nodes)) &
    (metric['kpi_name'].astype(str).str.contains(pattern, case=False, na=False)) &
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts)
].copy()

# Merge node_window with gp to evaluate anomalies
node_window = node_window.merge(gp_all, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark anomaly points: value > global_p95
node_window['is_anom'] = node_window['value'] > node_window['global_p95']

# Aggregate per (node, kpi)
agg_all = (
    node_window.groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(global_P95=('global_p95','first'),
         points_in_window=('value','count'),
         anomaly_count=('is_anom','sum'))
)

# earliest anomaly timestamp and max value within anomaly points (only from anomaly rows)
earliest_per = (
    node_window[node_window['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(earliest_anomaly_ts=('timestamp','min'),
         max_value_in_window=('value','max'))
)

# Merge earliest/max into agg_all
agg_all = agg_all.merge(earliest_per, on=['cmdb_id','kpi_name'], how='left')

# Compute severity_pct where applicable
def compute_severity(row):
    gp = row['global_P95']
    mv = row['max_value_in_window']
    if pd.isna(row['anomaly_count']) or int(row['anomaly_count']) == 0:
        return None
    if gp == 0 or pd.isna(gp) or pd.isna(mv):
        return None
    return (float(mv) - float(gp)) / float(gp) * 100.0

agg_all['severity_pct'] = agg_all.apply(compute_severity, axis=1)

# Convert earliest_anomaly_ts to UTC+8 datetime string
def ts_to_utc8_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

agg_all['earliest_anomaly_dt_utc8'] = agg_all['earliest_anomaly_ts'].apply(lambda x: ts_to_utc8_str(x) if pd.notna(x) else None)

# 4) Filter to rows with anomaly_count > 0
anoms = agg_all[agg_all['anomaly_count'] > 0].copy()

if anoms.empty:
    node_anomaly_summary = pd.DataFrame(
        [['NO_ANOMALIES_FOUND_FOR_NODES', None, None, None, None, None, None]],
        columns=['note','_1','_2','_3','_4','_5','_6']
    )
else:
    # Prepare compact output columns and sorting by severity_pct desc, tie-breaker earliest_anomaly_ts asc
    # Replace None severity_pct with -inf for sorting purposes so real numbers come first
    anoms['severity_sort'] = anoms['severity_pct'].apply(lambda x: -1e18 if x is None else x)
    anoms_sorted = anoms.sort_values(by=['severity_pct','earliest_anomaly_ts'], ascending=[False, True], na_position='last')
    # Select and rename columns per requirement
    node_anomaly_summary = anoms_sorted[[
        'cmdb_id','kpi_name','global_P95','points_in_window','anomaly_count',
        'earliest_anomaly_ts','earliest_anomaly_dt_utc8','max_value_in_window','severity_pct'
    ]].rename(columns={'cmdb_id':'node_cmdb_id'}).head(20).reset_index(drop=True)

# Display the compact result (full filtered table stored in node_window, and intermediate gp_all exists)
node_anomaly_summary

```
Out[7]:
```
Summary of node-level KPI anomaly scan for node-2, node-1, and node-6 (incident window 2022-03-20 18:30:00 → 18:59:00 UTC+8):

Key findings
- Multiple node-level KPIs on node-1, node-2 and node-6 show multi-point anomalies (anomaly_count = 2) within the incident window.
- The largest relative severities are on node-1 I/O metrics observed at ~18:34:
  - node-1: system.io.avg_q_sz — severity ≈ 1364% (max 0.52 vs global_P95 0.0355), earliest anomaly 2022-03-20 18:34:00.
  - node-1: system.io.await and system.io.w_await — severity ≈ 457% (max 2.81 vs global_P95 0.504), earliest 18:34:00.
  - node-1: system.io.w_s, system.cpu.user, system.cpu.iowait, system.cpu.system — also elevated (severity 126%–161%+), earliest anomalies around 18:34–18:45.
- node-2 shows significant I/O and CPU anomalies later in the window (18:41–18:47):
  - node-2: system.io.r_await, system.io.r_s, system.cpu.user, system.io.w_s, system.io.rkb_s, system.io.svctm, system.load.1 — severity ranging ~13% up to several hundred percent for some I/O KPIs; earliest anomaly timestamps between 18:41 and 18:47.
- node-6 has anomalies too (e.g., system.io.w_await and system.load.1) with moderate severity (~18%–24%), earliest around 18:37–18:48.
- Almost all flagged series have points_in_window = 30 and anomaly_count = 2 (i.e., two samples in the window exceeded the global P95).

Interpretation
- Strong signal of I/O subsystem stress (avg queue size, await, w_await, service time, r_s/w_s, rkb_s) and elevated CPU/iowait on node-1 early in the window (~18:34), followed by similar I/O/CPU signs on node-2 later (~18:41–18:47). Node-6 shows milder impact.
- The pattern suggests transient but substantial disk I/O contention impacting service responsiveness and CPU wait time across nodes—node-1 shows the most extreme relative deviations.

Suggested next steps
- Inspect system-level logs and disk metrics on node-1 around 18:34 (e.g., dmesg, iostat, disk error logs) to find root cause (heavy write/read burst, failing disk, noisy neighbor, backup job).
- Correlate with node-2 events around 18:41–18:47 to see whether the issue propagated or represents independent I/O bursts.
- Check impacted pods on those nodes for heavy disk I/O activity (containers doing backups, large writes, or checkpointing).
- If confirmed I/O saturation, consider throttling heavy jobs, moving IO-heavy pods, or investigating underlying storage performance.

Bottom line: node-1 exhibits the most severe I/O-related anomalies (huge relative spikes at ~18:34), node-2 shows significant I/O/CPU anomalies later, and node-6 has milder anomalies — prioritize node-1 I/O investigation first.

The original code execution output of IPython Kernel is also provided below for reference:

node_cmdb_id                 kpi_name  global_P95  points_in_window  anomaly_count  earliest_anomaly_ts earliest_anomaly_dt_utc8  max_value_in_window  severity_pct
0        node-1       system.io.avg_q_sz      0.0355                30              2         1.647772e+09  2022-03-20 18:34:00 CST                 0.52   1364.788732
1        node-1          system.io.await      0.5040                30              2         1.647772e+09  2022-03-20 18:34:00 CST                 2.81    457.539683
2        node-1        system.io.w_await      0.5040                30              2         1.647772e+09  2022-03-20 18:34:00 CST                 2.81    457.539683
3        node-2        system.io.r_await      0.5830                30              2         1.647773e+09  2022-03-20 18:44:00 CST                 2.50    328.816467
4        node-2            system.io.r_s     10.8750                30              2         1.647773e+09  2022-03-20 18:47:00 CST                37.50    244.827586
5        node-2          system.cpu.user      2.6315                30              2         1.647773e+09  2022-03-20 18:41:00 CST                 8.56    225.289759
6        node-1            system.io.w_s     35.5500                30              2         1.647772e+09  2022-03-20 18:34:00 CST                93.00    161.603376
7        node-1          system.cpu.user     11.5270                30              2         1.647773e+09  2022-03-20 18:45:00 CST                26.32    128.333478
8        node-1        system.cpu.iowait      1.7635                30              2         1.647772e+09  2022-03-20 18:34:00 CST                 3.99    126.254607
9        node-1     system.cpu.pct_usage     14.6145                30              2         1.647773e+09  2022-03-20 18:45:00 CST                27.82     90.358890
10       node-2            system.io.w_s    498.2000                30              2         1.647773e+09  2022-03-20 18:42:00 CST               783.50     57.266158
11       node-1        system.cpu.system      1.6315                30              2         1.647772e+09  2022-03-20 18:34:00 CST                 2.26     38.522832
12       node-2  system.fs.inodes.in_use      0.3300                30              1         1.647773e+09  2022-03-20 18:46:00 CST                 0.43     30.303030
13       node-6        system.io.w_await      1.1320                30              2         1.647773e+09  2022-03-20 18:37:00 CST                 1.40     23.674912
14       node-1          system.io.svctm      0.2255                30              2         1.647773e+09  2022-03-20 18:40:00 CST                 0.27     19.733925
15       node-1           system.io.util      0.8775                30              2         1.647773e+09  2022-03-20 18:51:00 CST                 1.05     19.658120
16       node-6            system.load.1      2.4925                30              2         1.647773e+09  2022-03-20 18:48:00 CST                 2.95     18.355065
17       node-2          system.io.rkb_s    235.7000                30              2         1.647773e+09  2022-03-20 18:47:00 CST               270.00     14.552397
18       node-2          system.io.svctm      0.8810                30              2         1.647773e+09  2022-03-20 18:45:00 CST                 1.00     13.507378
19       node-2            system.load.1      5.9870                30              2         1.647773e+09  2022-03-20 18:45:00 CST                 6.76     12.911308```
```